In [2]:
# !pip install fastapi
# !pip install uvicorn
# !pip install sentence-transformers
# !pip install faiss-cpu
# !pip install pydantic

In [3]:
# ==========================================================
# api/app.py
# ==========================================================

from fastapi import FastAPI
from pydantic import BaseModel

import faiss
import pickle
import pandas as pd

from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [4]:
app = FastAPI(
    title="Semantic Search API",
    description="Production Grade Semantic Search Engine",
    version="1.0.0",
)

In [5]:
print("Loading models...")

# Load SBERT
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Load Cross Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load FAISS
faiss_index = faiss.read_index("indexes/faiss_index.bin")

# Load Metadata
with open("indexes/metadata.pkl", "rb") as file:

    metadata = pickle.load(file)

print("API Ready!")

Loading models...


d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


API Ready!


In [6]:
class SearchRequest(BaseModel):
    query: str
    top_k: int = 5


class SearchResult(BaseModel):
    title: str
    category: str
    score: float


class SearchResponse(BaseModel):
    query: str
    results: list

In [7]:
def retrieve_candidates(query, top_k=5):

    query_embedding = bi_encoder.encode([query])

    distances, indices = faiss_index.search(query_embedding.astype("float32"), top_k)

    candidates = []

    for idx in indices[0]:

        doc = metadata[idx]

        candidates.append(doc)

    return candidates

In [8]:
def rerank_candidates(query, candidates):

    pairs = [(query, candidate["text"]) for candidate in candidates]

    scores = cross_encoder.predict(pairs)

    results = []

    for candidate, score in zip(candidates, scores):

        results.append(
            {
                "title": candidate["title"],
                "category": candidate["category"],
                "score": float(score),
            }
        )

    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return results

In [10]:
!uvicorn api.app:app --reload

: 

In [9]:
@app.get("/health")
def health_check():

    return {"status": "healthy", "service": "Semantic Search API"}